In [2]:
import os, math, random, io, requests, torch
import pandas as pd
import numpy as np
import evaluate
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix 
from transformers import pipeline, AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline
from datasets import Dataset, DatasetDict
from sklearn.utils.class_weight import compute_class_weight


2025-12-01 10:56:01.311687: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-01 10:56:01.311711: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-01 10:56:01.312906: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-01 10:56:01.318841: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Use NLI model for auto-labeling data to label data
- if justification aligns with the statement then it will be labeled as support
- if justifiticaiton does not align with statement then it will be deny


In [3]:
nli_ckpt = "roberta-large-mnli"
nli_pipe = pipeline("text-classification", model=nli_ckpt, tokenizer=nli_ckpt, return_all_scores=True, truncation=True)

def nli_to_stance(premise, hypothesis):
    # premise = justification; hypothesis = statement
    scores = nli_pipe([{"text": premise, "text_pair": hypothesis}])[0]
    # scores is list of dicts: {'label': 'ENTAILMENT'/'NEUTRAL'/'CONTRADICTION', 'score': prob}
    by_lbl = {d["label"].upper(): d["score"] for d in scores}
    pred = max(by_lbl, key=by_lbl.get)
    conf = by_lbl[pred]
    if pred == "ENTAILMENT": stance = "support"
    elif pred == "CONTRADICTION": stance = "deny"
    else: stance = "neutral"
    return stance, conf, by_lbl

# Label a dataframe; drop low-confidence
def pseudo_label(df, conf_thresh=0.6, max_rows=None):
    rows = df if max_rows is None else df.head(max_rows)
    stances, confs = [], []
    for prem, hyp in zip(rows["justification"], rows["statement"]):
        stance, conf, _ = nli_to_stance(prem, hyp)
        stances.append(stance); confs.append(conf)
    out = rows.copy()
    out["stance"] = stances
    out["stance_conf"] = confs
    out = out[out["stance_conf"] >= conf_thresh].reset_index(drop=True)
    return out

# pl_train = pseudo_label(train_df, conf_thresh=0.6, max_rows=None)
# pl_val   = pseudo_label(val_df,   conf_thresh=0.6, max_rows=None)
# pl_test  = pseudo_label(test_df,  conf_thresh=0.0, max_rows=None)  


config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/opt/conda/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
/opt/conda/lib/python3.11/site-packages/transformers/pipelines/text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [7]:
# the datasets are labeled by NFI by the code above
pl_train = pd.read_csv("./pl_train.csv")
pl_val   = pd.read_csv("./pl_val.csv")
pl_test  = pd.read_csv("./pl_test.csv")

In [8]:
# Balance the class, support, deny, and neutral by using the same size for training
min_count = pl_train["stance"].value_counts().min()
pl_train_bal = pl_train.groupby("stance").sample(min_count, random_state=42).reset_index(drop=True)
print("Balanced label counts:\n", pl_train_bal["stance"].value_counts())

Balanced label counts:
 stance
deny       367
neutral    367
support    367
Name: count, dtype: int64


In [9]:
# encode the label

unique_labels = sorted(pl_train_bal["stance"].unique().tolist())
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}
print("Label mapping:", label2id)

def encode_labels(df):
    df = df.copy()
    df["labels"] = df["stance"].map(label2id)
    return df

pl_train_enc = encode_labels(pl_train_bal)
pl_val_enc   = encode_labels(pl_val)
pl_test_enc  = encode_labels(pl_test)

def to_hf(df):
    return Dataset.from_pandas(df[["statement", "labels"]])


Label mapping: {'deny': 0, 'neutral': 1, 'support': 2}


In [10]:
# reformat dataset for training
raw_datasets = DatasetDict({
    "train": to_hf(pl_train_enc),
    "validation": to_hf(pl_val_enc),
    "test": to_hf(pl_test_enc)
})


In [11]:
# tokenize statement
model_ckpt = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_fn(batch):
    return tokenizer(batch["statement"], truncation=True, padding="max_length", max_length=512)

tokenized = raw_datasets.map(tokenize_fn, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/opt/conda/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/1101 [00:00<?, ? examples/s]

Map:   0%|          | 0/1016 [00:00<?, ? examples/s]

Map:   0%|          | 0/864 [00:00<?, ? examples/s]

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)
model.gradient_checkpointing_enable()  # saves VRAM

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
# model metrics
metric_acc = evaluate.load("accuracy")
metric_f1  = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=preds, references=labels)["accuracy"]
    macro_f1 = metric_f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "macro_f1": macro_f1}

In [14]:
# Define training configuration for HuggingFace Trainer

args = TrainingArguments(
    output_dir="./stance_distilroberta_final",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,         
    gradient_accumulation_steps=4,         
    per_device_eval_batch_size=4,
    fp16=True,                            
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir="/scratch/yul234/stance_logs_distilroberta",
)

/opt/conda/lib/python3.11/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [15]:
# train the model
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,1.064476,0.411417,0.397057
2,No log,1.095212,0.396654,0.374050
3,No log,1.115355,0.408465,0.393720


TrainOutput(global_step=207, training_loss=1.0053829616970487, metrics={'train_runtime': 112.7423, 'train_samples_per_second': 29.297, 'train_steps_per_second': 1.836, 'total_flos': 437547620662272.0, 'train_loss': 1.0053829616970487, 'epoch': 3.0})

In [16]:
# compare results to get model metrics
preds = trainer.predict(tokenized["test"])
y_true = preds.label_ids
y_pred = preds.predictions.argmax(-1)

print(classification_report(y_true, y_pred, target_names=unique_labels))



              precision    recall  f1-score   support

        deny       0.30      0.38      0.33       180
     neutral       0.71      0.36      0.48       482
     support       0.27      0.51      0.35       202

    accuracy                           0.40       864
   macro avg       0.42      0.42      0.39       864
weighted avg       0.52      0.40      0.42       864



In [17]:
# save model for export
trainer.save_model("./stance_distilroberta_final")
tokenizer.save_pretrained("./stance_distilroberta_final")

('./stance_distilroberta_final/tokenizer_config.json',
 './stance_distilroberta_final/special_tokens_map.json',
 './stance_distilroberta_final/vocab.json',
 './stance_distilroberta_final/merges.txt',
 './stance_distilroberta_final/added_tokens.json',
 './stance_distilroberta_final/tokenizer.json')

## test the model

In [18]:
stance_pipe = pipeline("text-classification", model="./stance_distilroberta_final/", tokenizer="./stance_distilroberta_final/", device=0)

text = pl_test.loc[pl_test["stance"] == "support", "statement"].iloc[2]
print(text)
print(stance_pipe(text, top_k=None))


Donald Trump is against marriage equality. He wants to go back.
[{'label': 'deny', 'score': 0.4507724642753601}, {'label': 'support', 'score': 0.3386603593826294}, {'label': 'neutral', 'score': 0.2105671465396881}]


In [19]:
article = """
The government, for example, demanded months ago that the University of California, Los Angeles, pay a settlement of more than $1 billion. School leaders have signaled that such a demand would be nearly impossible to meet.</p ><p class="css-ac37hb evys1bk0">The New York Times reported in June that Justice Department lawyers demanded that the University of Virginia oust Mr. Ryan to resolve an investigation into the school’s diversity, equity and inclusion efforts being led by Harmeet K. Dhillon, the department’s top civil rights lawyer, and Gregory Brown, the deputy assistant attorney general for civil rights. <a class="css-yywogo" href=" " title="">The following day, Mr. Ryan resigned</a >.</p ><p class="css-ac37hb evys1bk0">Democrats in Virginia have painted Mr. Ryan’s resignation as an act of capitulation. Republicans have pushed back on that notion, claiming that Mr. Ryan had been planning to leave and was a liability to the school because he was the wrong person to try to bring the school into line with the Trump administration’s agenda for higher education.</p ><p class="css-ac37hb evys1bk0">The University of Virginia’s potential agreement comes after the university surprised some administration officials by publicly rejecting a Trump administration proposal to link preferential treatment for federal funding to a school’s public commitment to President Trump’s higher education ideology.</p ><p class="css-ac37hb evys1bk0">Just hours after officials from several schools, including the University of Virginia, met with administration officials on Friday, the school’s interim president, Paul G. Mahoney, criticized the proposal for offering special treatment for some schools. He said the proposal, called the Compact for Academic Excellence in Higher Education, jeopardized “the integrity of science and other academic work requires merit-based assessment of research and scholarship.”</p ><p class="css-ac37hb evys1bk0">“A contractual arrangement predicating assessment on anything other than merit will undermine the integrity of vital, sometimes lifesaving, research and further erode confidence in American higher education,” Mr. Mahoney added.</p ></div>
"""

result = stance_pipe(article)
result


[{'label': 'deny', 'score': 0.3557637631893158}]

In [23]:
article = """against"""


result = stance_pipe(article)
result


[{'label': 'deny', 'score': 0.3744502663612366}]

In [20]:
id2label

{0: 'deny', 1: 'neutral', 2: 'support'}